In [1]:
import sqlite3
import pandas as pd
import time
from tqdm import tqdm

### Variables from conf file

In [3]:
# database file path
DB_FILE = "../../../drive_data/v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310.db"

# transaction table to update with status
TRANSACTION_TABLE = "transaction_row"

# column to show morph-syntax errors
STATUS_COL = "status"

### Connect to database

In [4]:
# connecting with database
conn = sqlite3.connect(DB_FILE)
cur = conn.cursor()

### Add new status column with empty strings

In [5]:
def column_exists(cursor, table, column):
    cursor.execute(f"PRAGMA table_info({table})")
    return any(row[1] == column for row in cursor.fetchall())

In [6]:
if not column_exists(cur, TRANSACTION_TABLE, STATUS_COL):
    cur.execute("ALTER TABLE " + TRANSACTION_TABLE +  f" ADD COLUMN {STATUS_COL} TEXT")

### Update status column based on rules

In [7]:
start = time.time()

# reset values to ensure reruns reflect updates
cur.execute(f"UPDATE {TRANSACTION_TABLE} SET {STATUS_COL} = '' ")

# RULES

# when nsubj is not in nominative/partitive case
cur.execute(f"""
UPDATE {TRANSACTION_TABLE}
SET {STATUS_COL} = 'syntax-morph conflict'
WHERE deprel = 'nsubj' 
        and ',' || feats || ',' NOT LIKE '%,nom,%' 
        and ',' || feats || ',' NOT LIKE '%,part,%'
""")

# when nsubj:cop is not in nominative/partitive case
cur.execute(f"""
UPDATE {TRANSACTION_TABLE}
SET {STATUS_COL} = 'syntax-morph conflict'
WHERE deprel = 'nsubj:cop' 
        and ',' || feats || ',' NOT LIKE '%,nom,%' 
        and ',' || feats || ',' NOT LIKE '%,part,%'
""")

# when obj is not in nominative/genitive/partitive case
cur.execute(f"""
UPDATE {TRANSACTION_TABLE}
SET {STATUS_COL} = 'syntax-morph conflict'
WHERE deprel = 'obj' 
        and ',' || feats || ',' NOT LIKE '%,nom,%'
        and ',' || feats || ',' NOT LIKE '%,gen,%' 
        and ',' || feats || ',' NOT LIKE '%,part,%'
""")


# when advcl has case marking
cur.execute(f"""
UPDATE {TRANSACTION_TABLE}
SET {STATUS_COL} = 'syntax-morph conflict'
WHERE deprel = 'advcl' 
        and feats!=''
""")

# when advmod has case marking
cur.execute(f"""
UPDATE {TRANSACTION_TABLE}
SET {STATUS_COL} = 'syntax-morph conflict'
WHERE deprel = 'advmod' 
        and feats!=''
""")


# when xcomp has case marking
cur.execute(f"""
UPDATE {TRANSACTION_TABLE}
SET {STATUS_COL} = 'syntax-morph conflict'
WHERE deprel = 'xcomp' 
        and feats!=''
""")


# when obl is in nominative case
cur.execute(f"""
UPDATE {TRANSACTION_TABLE}
SET {STATUS_COL} = 'syntax-morph conflict'
WHERE deprel = 'obl' 
        and ',' || feats || ',' LIKE '%,nom,%'
""")


conn.commit()

end = time.time()

elapsed_time = end - start

minutes = int(elapsed_time // 60)
seconds = int(elapsed_time % 60)

print(f"time: {minutes} minute(s) and {seconds} second(s)")

time: 3 minute(s) and 36 second(s)


### Check the results (not part of final workflow)

In [8]:
query = f"SELECT * FROM transaction_row limit 10"

res = pd.read_sql(query, conn)
res

,id,head_id,loc,loc_rel,deprel,form,lemma,feats,parent_loc,pos,status
0,1,2,3,-1,obl,lõpus,lõpp,"com,in,sg",None,S,
1,2,2,5,1,nsubj,Türi,Türi,"gen,prop,sg",None,S,syntax-morph conflict
2,3,2,6,2,obl,1.,1.,"<?>,ord,roman",None,N,
3,4,3,1,-3,obj,Bändi,bänd,"adit,com,sg",None,S,syntax-morph conflict
4,5,3,9,-2,nsubj,kidramees,kidramees,"com,nom,sg",None,S,
5,6,3,10,-1,aux,ei,ei,"aux,neg",None,V,
6,7,3,12,1,obl,keeltele,keel,"all,com,pl",None,S,
7,8,3,13,2,compound:prt,pihta,pihta,,None,D,
8,9,4,4,-2,nsubj,solist,solist,"com,nom,sg",None,S,
9,10,4,5,-1,aux,ei,ei,"aux,neg",None,V,


In [30]:
query = f"SELECT * FROM transaction_row where deprel='advcl' and status='syntax-morph conflict' limit 10"
# count 52665
res = pd.read_sql(query, conn)
res

,id,head_id,loc,loc_rel,deprel,form,lemma,feats,parent_loc,pos,status
0,29,17,11,2,advcl,viie-,viis,"card,gen,l,sg",None,N,syntax-morph conflict
1,68,37,4,-1,advcl,olles,olema,"ger,mod",None,V,syntax-morph conflict
2,77,46,9,1,advcl,hõivatud,hõivama,"aux,impf,imps,indic,neg",None,V,syntax-morph conflict
3,105,67,2,-1,advcl,näen,nägema,"af,aux,indic,pres,ps,ps1,sg",None,V,syntax-morph conflict
4,109,69,22,1,advcl,maksab,maksma,"af,aux,indic,pres,ps,ps3,sg",None,V,syntax-morph conflict
5,114,73,9,-1,advcl,ostes,ostma,"aux,ger",None,V,syntax-morph conflict
6,136,90,5,1,advcl,inspireerib,inspireerima,"af,indic,main,pres,ps,ps3,sg",None,V,syntax-morph conflict
7,137,92,4,-1,advcl,tulles,tulema,"aux,ger",None,V,syntax-morph conflict
8,139,92,8,2,advcl,arvates,arvama,"ger,mod",None,V,syntax-morph conflict
9,158,100,6,1,advcl,oska,oskama,"aux,indic,neg,pres,ps",None,V,syntax-morph conflict


In [ ]:
### ei tule sama error rate nagu tabelis???

In [22]:
query = f"SELECT count(DISTINCT head_id) FROM transaction_row where deprel = 'obl'"

res = pd.read_sql(query, conn)
res

,count(DISTINCT head_id)
0,9308925


In [17]:
query = f"SELECT count(*) FROM transaction_row where deprel='obl' and status='syntax-morph conflict'"

res = pd.read_sql(query, conn)
res

,count(*)
0,325406


In [24]:
(325406/9308925) * 100000

3495.6345657527586

In [9]:
conn.close()